# Fine-Tune Llama 3.2 3B for Lordy's Assistant
**Pure PEFT + Transformers - NO Unsloth**
Runtime -> Change runtime type -> T4 GPU


In [ ]:
# Cell 1: Install packages (no Unsloth)
!pip install -q transformers peft accelerate bitsandbytes trl datasets
print("Done")


In [ ]:
# Cell 2: Load Llama 3.2 3B in 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
print("Model loaded")


In [ ]:
# Cell 3: Add LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# Cell 4: Training data
from datasets import Dataset

train_data = [
    {"instruction": "Analyze betting odds for Argentina vs Egypt at 1.27",
     "output": "Argentina at 1.27 gives ~70% win probability. World Champions with Messi, Alvarez, Martinez. $100 bet pays $127."},
    {"instruction": "What is the win rate for Morocco at odds 1.77?",
     "output": "Morocco at 1.77 gives ~51% win probability. Below 70% threshold but favored. Canada weaker but still competitive."},
    {"instruction": "What are good bets on this screen? Argentina 1.27, Colombia 1.35, Brazil 1.63",
     "output": "Argentina 1.27 (70%) is safest. Colombia 1.35 (65%) is good value. Brazil 1.63 (54%) is risky."},
    {"instruction": "Plan Singapore to Macau to cold China city in December",
     "output": "From Macau: Beijing (3hr, -5C snow), Xian (2.5hr, $50-100, 0-8C), Harbin (4hr, -20C). Best: Beijing."},
    {"instruction": "What hotels near KSL Mall Johor Bahru?",
     "output": "KSL Resort (4-star, $66/night, connected to mall, pool), Fives Hotel ($38, 3min walk), Grand Paragon ($35, 5min drive)."},
    {"instruction": "What is my passive income from Nexo?",
     "output": "$93,000 USDT at 8.25% APY = $7,672/year or $639/month. No lockup."},
    {"instruction": "What train from Kansai Airport to Namba Osaka?",
     "output": "Nankai Rapi:t limited express - 34-38 min, 1,400 yen direct to Namba Station."},
    {"instruction": "What should I buy in Japan?",
     "output": "Medicines: Pabron Gold A, Bufferin, Salonpas, Rohto eye drops. Skincare: Anessa sunscreen, Hada Labo. Snacks: unique Kit Kat flavors."},
    {"instruction": "Estimate USDT interest on $93,000 at 8.25% for 1 month",
     "output": "$93,000 x 8.25% / 12 = $639/month or $21.30/day passive income."},
    {"instruction": "Compare DeepSeek V4 Flash vs Llama 3.2 3B",
     "output": "DeepSeek V4 Flash is 600B MoE model, costs $0.14/M tokens. Llama 3.2 3B is smaller, free local, good for offline tasks."},
    {"instruction": "Is my Mac mini good for local AI?",
     "output": "Intel i5-8500B with 8GB RAM. Can run 1-3B models. 7B+ is slow. Use DeepSeek API for heavy work, Llama 3.2 3B for offline."},
    {"instruction": "What should I do with my $97K in crypto?",
     "output": "$93K USDT at 8.25% on Nexo ($639/mo) + $4K NEXO. Good passive income. Consider Binance USD1 flex (10.5%) for better rate."},
]

def fmt(entry):
    return f"### Instruction:\n{entry['instruction']}\n\n### Response:\n{entry['output']}{tokenizer.eos_token}"

dataset = Dataset.from_list([{"text": fmt(d)} for d in train_data])
print(f"Training on {len(dataset)} examples")


In [ ]:
# Cell 5: TRAIN! (Pure HF Trainer)
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize
tok = dataset.map(lambda x: tokenizer(x["text"], truncation=True, max_length=2048))

trainer = Trainer(
    model=model,
    train_dataset=tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()
print("TRAINING COMPLETE!")


In [ ]:
# Cell 6: Save LoRA adapter
model.save_pretrained("lordy-lora")
tokenizer.save_pretrained("lordy-lora")
print("LoRA adapter saved to lordy-lora/")
print()
print("Download the lordy-lora folder to your Mac")
